<a href="https://colab.research.google.com/github/skshahid0786/0x44/blob/main/SmolLM_135M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install & Import
!pip install -q -U trl transformers accelerate datasets
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

# 2. Load the data and RENAME the columns to match what the AI expects
dataset = load_dataset("nvidia/HelpSteer", split="train[:1000]")

# This fixes the 'KeyError' by giving the AI 'prompt' and 'completion'
dataset = dataset.rename_column("response", "completion")

# 3. The Professional Trainer
trainer = SFTTrainer(
    model="HuggingFaceTB/SmolLM2-135M",
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="prompt",  # The user question
        max_length=512,
        output_dir="./jarvis-final",
        per_device_train_batch_size=2,
        num_train_epochs=1,
        learning_rate=5e-5,
        report_to="none"
    ),
)

trainer.train()
trainer.save_model("./jarvis-final")
print("Master Shahid, Jarvis is now truly smart and ready!")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Step,Training Loss
10,1.400469
20,1.104979
30,1.382052
40,2.404453
50,2.182975
60,1.798075
70,1.322264
80,1.324006
90,1.302730
100,1.712029


## Local Inference on GPU
Model page: https://huggingface.co/HuggingFaceTB/SmolLM-135M

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/HuggingFaceTB/SmolLM-135M)
            and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:

import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 1. Manually point to your folder
# Change this if your folder name in the files tab is different!
model_path = "./jarvis-pro-final"

print("Loading local brain... please wait.")

# 2. Load the pieces manually to avoid the HF Error
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    local_files_only=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. Put it in the chat pipe
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def chat_with_jarvis(message, history):
    system_rules = (
        "You are JARVIS, Shahid's professional AI. "
        "Answer briefly and politely. No inappropriate topics."
    )
    prompt = f"System: {system_rules}\nUser: {message}\nAssistant:"

    output = pipe(prompt, max_new_tokens=60, do_sample=True, temperature=0.3, repetition_penalty=1.4)
    response = output[0]['generated_text'].split("Assistant:")[-1].strip()
    return response.split("User:")[0].strip()

demo = gr.ChatInterface(fn=chat_with_jarvis, title="J.A.R.V.I.S. PRO")
demo.launch(share=True)